# 제출용 파일 3/3 · 6멤버 랭크 블렌드 → `submission_final.csv`

파일 1(케글, 5멤버)·파일 2(TabM·전처리 통일)가 저장한 `oof_*`·`test_*` 6쌍을 로드해
**랭크(rank) 블렌딩**으로 최종 `submission_final.csv`를 생성합니다. (CPU 어디서나, 수초.)

**최종 모델 — 6멤버 랭크 블렌드** · Public LB **0.7424857289**
`lgb(v2v3) · cat(v2v3) · xgb(v3) · lin(ratio) · nn(v2v3) · tabm(전처리 통일)` · 가중치는 OOF(train 라벨만) 힐클라이밍.
산출 가중치 `{lgb 0.20, cat 0.212, xgb 0.094, lin 0.012, nn 0.141, tabm 0.341}` · 블렌드 OOF 0.74090.

**왜 랭크 블렌딩인가 — 딥 비결정성 방어 장치**
- 트리·선형은 완전 결정적이지만 NN·TabM은 하드웨어 의존으로 확률이 소수점 4째자리에서 흔들립니다.
- 확률 평균 대신 **순위(rank) 평균**을 쓰면, 멤버 확률이 미세히 변해도 *순위*는 거의 불변 → **AUC가 보존**됩니다. 비결정성을 없애지 못하는 대신, 블렌드 구조로 흡수합니다.

**입력:** `oof_{lgb,cat,xgb,lin,nn,tabm}.csv` · `test_{...}.csv` (파일 1·2 산출)
**산출:** `submission_final.csv` (+ 보너스 `submission_no_lin.csv`) · `reproduce_report.json`

In [29]:
import os, glob, json, sys
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

# 0. 재현성 고정
SEED = 42
np.random.seed(SEED)

def runr(x): return rankdata(x) / len(x)

def find_csv(n):
    # 현재 폴더와 하위 폴더들을 전부 뒤져서 파일을 자동으로 찾아줍니다.
    for d in [".", "data", "../data", "/kaggle/input", "/kaggle/working"]:
        p = os.path.join(d, n)
        if os.path.exists(p): return p
    h = [p for p in glob.glob(f"**/{n}", recursive=True)]
    return sorted(h, key=len)[0] if h else None

# 1. 원본 데이터의 ID 및 기준 라벨(y) 로드
_ref_test = find_csv("test.csv") or "/kaggle/input/datasets/yerim0722/data123/test.csv"
test_ids = pd.read_csv(_ref_test)["ID"].values if _ref_test else None

MEMBERS = ["lgb", "cat", "xgb", "lin", "nn", "tabm"]
members_oof = {}
members_test = {}
yref = None

print("--- 1. 6멤버 예측 결과 로드 시작 ---")
for m in MEMBERS:
    op = find_csv(f"oof_{m}.csv")
    tp = find_csv(f"test_{m}.csv")
    
    if not op or not tp:
        raise FileNotFoundError(f"🚨 {m} 모델의 oof 또는 test 파일을 찾을 수 없습니다! 파일1, 2의 결과물과 동일한 폴더에 있는지 확인하세요.")
        
    od = pd.read_csv(op)
    td = pd.read_csv(tp)
    
    o = od[f"oof_{m}"].values
    if "y" in od.columns:
        yy = od["y"].astype(int).values
        yref = yy if yref is None else yref
        assert np.array_equal(yy, yref), f"🚨 {m}: oof의 y(정답)가 다른 멤버와 다릅니다!"
        
    if test_ids is not None and "ID" in td.columns:
        t = td.set_index("ID")[f"test_{m}"].reindex(test_ids).values
    else:
        t = td[f"test_{m}"].values
        
    a = roc_auc_score(yref, o)
    members_oof[m] = o
    members_test[m] = t
    print(f"  ✅ [로드 완료] {m:5s} OOF AUC = {a:.5f}")

y = yref

# 2. 랭크 블렌딩 (힐 클라이밍 최적화)
print("\n--- 2. 힐 클라이밍 랭크 블렌딩 시작 ---")
def hill(d, yy, n=120):
    nm = list(d)
    s0 = {k: roc_auc_score(yy, d[k]) for k in nm}
    b = max(s0, key=s0.get)
    ens = [b]
    s = d[b].copy()
    best = (list(ens), s0[b])
    
    for _ in range(n):
        cb, ca = None, -1
        for k in nm:
            a = roc_auc_score(yy, (s + d[k]) / (len(ens) + 1))
            if a > ca: 
                ca, cb = a, k
        ens.append(cb)
        s = s + d[cb]
        if ca > best[1]: 
            best = (list(ens), ca)
            
    from collections import Counter
    c = Counter(best[0])
    return {k: c.get(k, 0) / len(best[0]) for k in nm}, best[1]

R = {m: runr(members_oof[m]) for m in members_oof}
w, blend_oof = hill(R, y)

print(f"🌟 최종 6멤버 블렌드 OOF AUC = {blend_oof:.5f}")
print("🌟 산출된 최적 가중치:", {k: round(v, 3) for k, v in w.items()})

# 3. 제출 파일 생성
p_test = sum(w[m] * runr(members_test[m]) for m in w if w[m] > 0)
sub = pd.DataFrame({"ID": test_ids if test_ids is not None else range(len(p_test)), "probability": p_test})

# sample_submission.csv 양식에 맞추기
sp = find_csv("sample_submission.csv") or "/kaggle/input/datasets/yerim0722/data123/sample_submission.csv"
if sp:
    s = pd.read_csv(sp)
    pc = [c for c in s.columns if c.lower() != "id"][0]
    s[pc] = p_test
    sub = s

sub.to_csv("submission_final.csv", index=False)
print(f"\n💾 최종 제출 파일 저장 완료: submission_final.csv (데이터 수: {len(sub)})")

# 4. [추가 검증] LIN 제외 테스트 (가중치가 너무 낮아 노이즈가 될 수 있는지 검증)
print("\n--- [보너스 검증] LIN 모델 제외 시 성능 검증 ---")
R_no_lin = {m: runr(members_oof[m]) for m in members_oof if m != "lin"}
w_no_lin, blend_oof_no_lin = hill(R_no_lin, y)

print(f"  LIN 제외 OOF = {blend_oof_no_lin:.5f} / 6멤버 전체 OOF = {blend_oof:.5f}")
if blend_oof_no_lin >= blend_oof - 0.00005:
    print("  💡 LIN을 빼는 것이 OOF가 유지되거나 더 개선됩니다. (submission_no_lin.csv 저장)")
    p_test_no_lin = sum(w_no_lin[m] * runr(members_test[m]) for m in w_no_lin if w_no_lin[m] > 0)
    sub_no_lin = sub.copy()
    sub_no_lin[sub_no_lin.columns[-1]] = p_test_no_lin
    sub_no_lin.to_csv("submission_no_lin.csv", index=False)
else:
    print("  💡 LIN이 작지만 긍정적인 기여를 하고 있습니다. submission_final.csv를 우선 제출하세요.")

--- 1. 6멤버 예측 결과 로드 시작 ---
  ✅ [로드 완료] lgb   OOF AUC = 0.73965
  ✅ [로드 완료] cat   OOF AUC = 0.73974
  ✅ [로드 완료] xgb   OOF AUC = 0.73964
  ✅ [로드 완료] lin   OOF AUC = 0.72029
  ✅ [로드 완료] nn    OOF AUC = 0.73812
  ✅ [로드 완료] tabm  OOF AUC = 0.74002

--- 2. 힐 클라이밍 랭크 블렌딩 시작 ---
🌟 최종 6멤버 블렌드 OOF AUC = 0.74090
🌟 산출된 최적 가중치: {'lgb': 0.2, 'cat': 0.212, 'xgb': 0.094, 'lin': 0.012, 'nn': 0.141, 'tabm': 0.341}

💾 최종 제출 파일 저장 완료: submission_final.csv (데이터 수: 90067)

--- [보너스 검증] LIN 모델 제외 시 성능 검증 ---
  LIN 제외 OOF = 0.74090 / 6멤버 전체 OOF = 0.74090
  💡 LIN을 빼는 것이 OOF가 유지되거나 더 개선됩니다. (submission_no_lin.csv 저장)
